In [1]:
import sys
from pathlib import Path

# 프로젝트 루트를 path에 추가 (노트북이 juypter-notebook/ 또는 juypter-notebook/app/ 에 있을 때)
_root = (
    Path.cwd().parent
    if (Path.cwd().parent / "document_parsing_engine").exists()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(_root))

from document_parsing_engine.app.services.document_classification_service import (
    DocumentClassificationService,
)
from document_parsing_engine.app.services.document_layout_parsing_service import (
    DocumentLayoutParsingService,
)
from document_parsing_engine.app.services.layout_segment_mapping_service import (
    LayoutSegmentMappingService,
)
from document_parsing_engine.app.services.field_mapping_service import (
    FieldMappingService,
)
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
pdf_path = Path(
    "../../1_quotation_sample/27188_1.견적요청서.pdf"
)  # app/ 에서는 상위 2단계
if not pdf_path.exists():
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {pdf_path}")


# 3. layout parsing
# blocks = layout_parser.parse_layout(doc_dict)
# # 4. segment mapping
# seg = segment_mapper.recommend(doc_type=classification.doc_type.value, doc_dict=doc_dict, blocks=blocks)
# # 5. field mapping (ANTHROPIC_API_KEY 필요)
# field_result = field_mapper.extract(classification.doc_type.value, seg, blocks)

# print("doc_type:", classification.doc_type)
# print("field_result keys:", list(field_result.keys()))
# field_result

In [2]:
result = converter.convert(pdf_path)
doc_dict = result.document.export_to_dict()

# 서비스 인스턴스
classifier = DocumentClassificationService()
layout_parser = DocumentLayoutParsingService()
segment_mapper = LayoutSegmentMappingService()
field_mapper = FieldMappingService()

# 2. 문서 타입 분류
classification = classifier.classify(doc_dict)

In [3]:
doc_dict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [4]:
classification

ClassificationResult(doc_type=<DocType.QUOTATION: 'quotation'>, score=4, reasons=['text contains "견적": "자재/부품 견적요청서"', 'table header "규격": "재질 및 규격"'], all_scores={'quotation': 4})

In [5]:
blocks = layout_parser.parse_layout(doc_dict)
blocks

[BlockContent(ref='#/texts/0', label='page_header', content_type='text', content='DOC. NO. : KTL_KDC_002'),
 BlockContent(ref='#/texts/1', label='section_header', content_type='text', content='자재/부품 견적요청서'),
 BlockContent(ref='#/pictures/0', label='picture', content_type='picture', content=None),
 BlockContent(ref='#/tables/0', label='table', content_type='table', content=[['1. 견적의뢰정보', '1. 견적의뢰정보', '2.', '의뢰자정보', '의뢰자정보', '의뢰자정보', '3. 필수요청사항', '3. 필수요청사항', '3. 필수요청사항'], ['상 호', '마켓오브메테리얼', '상', '호', '케이티엘㈜', '', '1. 세금계산서 발행메일 info@tkdk.co.kr', '', ''], ['수 신', '김경호 이사님', '', '사업자등록번호', '621-81-36401 / 대표 : 지춘현', '621-81-36401 / 대표 : 지춘현', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점'], ['견적요청일', '2024년 02월 28일', '', '담당자', '지성민', '지성민', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166'], ['납품요청일', '납품일 별도 회신 요청', '담당자', '연락처', '051-532-0471 / Sales2@tkdk.co.kr', '051-532-0471 / Sales2@tkdk.co.k

In [6]:
layout_parser.print_blocks_content(doc_dict)


[0] #/texts/0  label=page_header
DOC. NO. : KTL_KDC_002

[1] #/texts/1  label=section_header
자재/부품 견적요청서

[2] #/pictures/0  label=picture
    [PICTURE]

[3] #/tables/0  label=table
    row[0] = ['1. 견적의뢰정보', '1. 견적의뢰정보', '2.', '의뢰자정보', '의뢰자정보', '의뢰자정보', '3. 필수요청사항', '3. 필수요청사항', '3. 필수요청사항']
    row[1] = ['상 호', '마켓오브메테리얼', '상', '호', '케이티엘㈜', '', '1. 세금계산서 발행메일 info@tkdk.co.kr', '', '']
    row[2] = ['수 신', '김경호 이사님', '', '사업자등록번호', '621-81-36401 / 대표 : 지춘현', '621-81-36401 / 대표 : 지춘현', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점']
    row[3] = ['견적요청일', '2024년 02월 28일', '', '담당자', '지성민', '지성민', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166']
    row[4] = ['납품요청일', '납품일 별도 회신 요청', '담당자', '연락처', '051-532-0471 / Sales2@tkdk.co.kr', '051-532-0471 / Sales2@tkdk.co.kr', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail

In [7]:
seg = segment_mapper.recommend(
    doc_type=classification.doc_type.value, doc_dict=doc_dict, blocks=blocks
)

In [8]:
segment_mapper.pretty_print(seg, blocks)

=== segment: items ===
  block_refs: ['#/tables/0']
  reasons: ['field keywords matched: 재질, 규격', 'segment keywords matched: 재질, 규격', 'content_type=table']...
    - #/tables/0 | label=table | content_type=table
       ['1. 견적의뢰정보', '1. 견적의뢰정보', '2.', '의뢰자정보', '의뢰자정보', '의뢰자정보', '3. 필수요청사항', '3. 필수요청사항', '3. 필수요청사항']
       ['상 호', '마켓오브메테리얼', '상', '호', '케이티엘㈜', '', '1. 세금계산서 발행메일 info@tkdk.co.kr', '', '']
       ['수 신', '김경호 이사님', '', '사업자등록번호', '621-81-36401 / 대표 : 지춘현', '621-81-36401 / 대표 : 지춘현', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점']
       ['견적요청일', '2024년 02월 28일', '', '담당자', '지성민', '지성민', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166']
       ['납품요청일', '납품일 별도 회신 요청', '담당자', '연락처', '051-532-0471 / Sales2@tkdk.co.kr', '051-532-0471 / Sales2@tkdk.co.kr', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail 

## 세그먼트 분해
* 마지막 단계에 넣자

In [21]:
seg.allowed_segments

['quotation_meta', 'items', 'remarks']

In [22]:
seg.segment_buckets

[SegmentBucket(segment_name='items', block_refs=['#/tables/0'], reasons=['field keywords matched: 재질, 규격', 'segment keywords matched: 재질, 규격', 'content_type=table', 'label=table'])]

In [23]:
def segments_without_buckets(seg) -> list[str]:
    """allowed_segments와 segment_buckets를 비교해, 버킷이 없는(매핑된 블록이 없는) 세그먼트 목록을 반환."""
    allowed = set(seg.allowed_segments)
    bucket_names = {b.segment_name for b in seg.segment_buckets}
    return sorted(allowed - bucket_names)


missing = segments_without_buckets(seg)
if missing:
    print("버킷이 없는 세그먼트:", missing)
else:
    print("모든 allowed_segments에 대응하는 segment_bucket이 있음.")
missing

버킷이 없는 세그먼트: ['quotation_meta', 'remarks']


['quotation_meta', 'remarks']

### 1. 이는 items에 모든게 합쳐진 가능성이 크다
* 표로 작성하는 경우가 많기 때문

### 2. OCR로 읽어야함

In [4]:
result = converter.convert(
    "../../1_quotation_sample/36572_Att _2 Quotation_Cuni Pipe_Fitting_20230412 -2_MOM.PDF"
)
doc_dict = result.document.export_to_dict()
classification = classifier.classify(doc_dict)
print(classification)
# blocks = layout_parser.parse_layout(doc_dict)
# seg = segment_mapper.recommend(
#     doc_type=classification.doc_type.value, doc_dict=doc_dict, blocks=blocks
# )
# segment_mapper.pretty_print(seg, blocks)

ClassificationResult(doc_type=<DocType.QUOTATION: 'quotation'>, score=14, reasons=['title exact match(section_header): "Q U O T E"', 'table header "unit price": "unit price"', 'table header "amount": "amount"'], all_scores={'quotation': 14, 'invoice': 6})


In [7]:
def _looks_like_docling_internal_ref(text) -> bool:
    """Docling export_to_dict()가 내부 참조(/18, /i255 등)로 넣었는지 판별."""
    if not text or not isinstance(text, str):
        return False
    s = text.strip()
    if not s.startswith("/"):
        return False
    return any(c.isdigit() for c in s) or "i" in s[:5]


def print_doc_dict_content(
    doc_dict, max_text_len=200, max_table_rows=20, docling_document=None
):
    """doc_dict 구조를 따라가며 텍스트/테이블/그룹 등 컨텐츠를 출력.
    docling_document가 주어지면, 텍스트가 내부 참조(/18, /i255 등)일 때
    Docling API(export_to_text)로 실제 문자열을 가져와 출력한다.
    """
    from document_parsing_engine.domain.layout.doc_blocks import get_blocks_sorted

    blocks_sorted = get_blocks_sorted(doc_dict)
    for i, (ref, obj) in enumerate(blocks_sorted):
        label = obj.get("label", "")
        text = obj.get("text") or obj.get("orig", "")
        if text:
            s = str(text).strip()
            # Docling이 내부 참조로 저장한 경우, docling_document로 실제 텍스트 조회
            if docling_document and _looks_like_docling_internal_ref(s):
                try:
                    resolved = docling_document.export_to_text(
                        from_element=i, to_element=i + 1
                    ).strip()
                    if resolved:
                        s = resolved
                except Exception:
                    s = f"(내부 참조, 해석 실패: {s})"
            if len(s) > max_text_len:
                s = s[:max_text_len] + "..."
            print(f"[{i}] {ref}  label={label}")
            print(f"    {s}")
            print()
            continue
        if "rows" in obj or (
            isinstance(obj.get("content"), list) and obj.get("content")
        ):
            rows = obj.get("rows") or obj.get("content", [])
            print(f"[{i}] {ref}  label={label} (table, {len(rows)} rows)")
            for ri, row in enumerate(rows[:max_table_rows]):
                print(f"    row[{ri}] = {row}")
            if len(rows) > max_table_rows:
                print(f"    ... ({len(rows) - max_table_rows} more rows)")
            print()
            continue
        if obj.get("children"):
            print(
                f"[{i}] {ref}  label={label} (group, {len(obj['children'])} children)"
            )
            print()
        else:
            print(f"[{i}] {ref}  label={label}")
            print()


print_doc_dict_content(doc_dict)

NameError: name 'doc_dict' is not defined

In [8]:
blocks = layout_parser.parse_layout(doc_dict)
layout_parser.print_blocks_content(doc_dict)

NameError: name 'layout_parser' is not defined

In [7]:
seg = segment_mapper.recommend(
    doc_type=classification.doc_type.value, doc_dict=doc_dict, blocks=blocks
)
segment_mapper.pretty_print(seg, blocks)

=== segment: quotation_meta ===
  block_refs: ['#/tables/0']
  reasons: ['field keywords matched: no., to, Reference, payment, delivery', 'segment keywords matched: quote', 'content_type=table']...
    - #/tables/0 | label=table | content_type=table
       ['Quote No.', '2023/04/12 -2']
       ['To', '피오르드프로세싱코리아 주식회사']
       ['Reference', 'Ms. Minju Oh']
       ['Project', 'NOV - Cu-Ni Pipe (GALLAF']
       ['Payment To be discussed', '']
       ['Valid Until', '견적 제출 후 3일']
       ['Delivery Terms', 'EXW UK']
       ['Delivery', 'REMARK 참고']

=== segment: items ===
  block_refs: ['#/tables/1', '#/tables/2', '#/tables/3']
  reasons: ['field keywords matched: material', 'content_type=table', 'label=table']...
    - #/tables/1 | label=table | content_type=table
       ['Company', 'MARKET OF MATERIAL']
       ['114-301, 50, UNIST-gil, Eonyang-eup, Ulju-gun, Ulsan, Republic of Korea', 'Address']
       ['+82 10-9270-3873', 'TEL.']
       ['Kayne Kim', 'Sales Rep.']
       ['E-Mail kyeong

### 재구성 1. 여러 테이블이 하나의 세그먼트에 모여있을때

In [9]:
# 재구성 1 또는 재구성 2 한 번에 적용 (서비스)
# 버킷 없는 세그먼트가 없으면 seg, blocks 그대로 반환.
# 다수 테이블이 한 세그먼트에 모여 있으면 재구성 1, 테이블 하나면 2.1 행분해 후 2.2 열분해+remarks 적용.

from document_parsing_engine.app.services.segment_reconstruction_service import reconstruct

refined_seg, refined_blocks = reconstruct(seg, blocks)
segment_mapper.pretty_print(refined_seg, refined_blocks)

=== segment: quotation_meta ===
  block_refs: ['#/tables/0@quotation_meta']
    - #/tables/0@quotation_meta | label=table | content_type=table
       ['견적의뢰정보', '의뢰자정보']
       ['상 호 마켓오브메테리얼', '상 호 케이티엘㈜']
       ['수 신 김경호 이사님', '사업자등록번호 621-81-36401 / 대표 : 지춘현']
       ['견적요청일 2024년 02월 28일', '담당자 지성민']
       ['납품요청일 납품일 별도 회신 요청', '담당자 연락처 051-532-0471 / Sales2@tkdk.co.kr']
       ['특이사항', '특이사항']
       ['자재/부품 요청내역', '자재/부품 요청내역']
       ['최 종 합 계 - ₩', '최 종 합 계 - ₩']

=== segment: items ===
  block_refs: ['#/tables/0@items']
    - #/tables/0@items | label=table | content_type=table
       ['NO.', '품 목', '재질 및 규격', '수 량', '단 위', '단 가', '합 계']
       ['1', 'Flange', 'ASEM #150 A105 Sch.40 WN.RF 4" (MFG : 펠릭스테크)', '16', 'EA', '', '']
       ['2', '', '', '', '', '', '']
       ['3', '', '', '', '', '', '']
       ['4', '', '', '', '', '', '']
       ['5', '', '', '', '', '', '']
       ['6', '', '', '', '', '', '']
       ['7', '', '', '', '', '', '']
       ['8', '', '', '', '', '

In [6]:
seg = segment_mapper.recommend(
    doc_type=classification.doc_type.value, doc_dict=doc_dict, blocks=blocks
)
segment_mapper.pretty_print(seg, blocks)

NameError: name 'segment_mapper' is not defined

### 재구성 2. 표가 하나로 합쳐져있을때

In [ ]:
result = converter.convert("../../1_quotation_sample/27188_1.견적요청서.pdf")
doc_dict = result.document.export_to_dict()
classification = classifier.classify(doc_dict)
print(classification)

ClassificationResult(doc_type=<DocType.QUOTATION: 'quotation'>, score=4, reasons=['text contains "견적": "자재/부품 견적요청서"', 'table header "규격": "재질 및 규격"'], all_scores={'quotation': 4})


In [17]:
blocks = layout_parser.parse_layout(doc_dict)
layout_parser.print_blocks_content(doc_dict)


[0] #/texts/0  label=page_header
DOC. NO. : KTL_KDC_002

[1] #/texts/1  label=section_header
자재/부품 견적요청서

[2] #/pictures/0  label=picture
    [PICTURE]

[3] #/tables/0  label=table
    row[0] = ['1. 견적의뢰정보', '1. 견적의뢰정보', '2.', '의뢰자정보', '의뢰자정보', '의뢰자정보', '3. 필수요청사항', '3. 필수요청사항', '3. 필수요청사항']
    row[1] = ['상 호', '마켓오브메테리얼', '상', '호', '케이티엘㈜', '', '1. 세금계산서 발행메일 info@tkdk.co.kr', '', '']
    row[2] = ['수 신', '김경호 이사님', '', '사업자등록번호', '621-81-36401 / 대표 : 지춘현', '621-81-36401 / 대표 : 지춘현', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점']
    row[3] = ['견적요청일', '2024년 02월 28일', '', '담당자', '지성민', '지성민', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166']
    row[4] = ['납품요청일', '납품일 별도 회신 요청', '담당자', '연락처', '051-532-0471 / Sales2@tkdk.co.kr', '051-532-0471 / Sales2@tkdk.co.kr', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail

### 2.1 행 분해 (아이템 추출)


In [ ]:
# 재구성 2.1 행 분해 (아이템 추출)
# 참고: reconstruct()가 테이블 하나일 때 자동으로 2.1 → 2.2 순서 호출.

from document_parsing_engine.domain.segment_reconstruction import decompose_merged_tables_by_row_sections

refined_seg, refined_blocks = decompose_merged_tables_by_row_sections(seg, blocks)
segment_mapper.pretty_print(refined_seg, refined_blocks)

=== segment: quotation_meta ===
  block_refs: ['#/tables/0@quotation_meta']
    - #/tables/0@quotation_meta | label=table | content_type=table
       ['1. 견적의뢰정보', '1. 견적의뢰정보', '2.', '의뢰자정보', '의뢰자정보', '의뢰자정보', '3. 필수요청사항', '3. 필수요청사항', '3. 필수요청사항']
       ['상 호', '마켓오브메테리얼', '상', '호', '케이티엘㈜', '', '1. 세금계산서 발행메일 info@tkdk.co.kr', '', '']
       ['수 신', '김경호 이사님', '', '사업자등록번호', '621-81-36401 / 대표 : 지춘현', '621-81-36401 / 대표 : 지춘현', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점', '2. 화물 납품 대신화물/택배 부산장안점']
       ['견적요청일', '2024년 02월 28일', '', '담당자', '지성민', '지성민', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166']
       ['납품요청일', '납품일 별도 회신 요청', '담당자', '연락처', '051-532-0471 / Sales2@tkdk.co.kr', '051-532-0471 / Sales2@tkdk.co.kr', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행

### 2.2 열분해 (qutation_meta, remarks 추출)


In [34]:
# 참고: 아래 로직은 domain/segment_reconstruction/column_merge.py에 있으며 reconstruct() 호출 시 자동 적용됩니다.
import re


def _clean_section_cell(cell) -> str:
    """셀에서 앞쪽 '숫자.' 또는 '숫자. ' 제거 후 반환."""
    if cell is None:
        return ""
    s = str(cell).strip()
    return re.sub(r"^\d+\.\s*", "", s).strip()


def _runs_of_same(cleaned: list[str]) -> list[tuple[str, int]]:
    """연속된 같은(비공란) 값만 (값, 연속 개수) 리스트로."""
    runs = []
    for c in cleaned:
        if not c:
            continue
        if runs and runs[-1][0] == c:
            runs[-1] = (c, runs[-1][1] + 1)
        else:
            runs.append((c, 1))
    return runs


def normalize_and_merge_section_row(row: list, cells_per_run: int = 2) -> list[str]:
    """
    - 일반화: 각 셀에서 앞 '숫자.' 제거.
    - 병합: 연속된 같은 값은 한 run으로 보고, run당 cells_per_run개 셀로 내보냄.
    """
    cleaned = [_clean_section_cell(c) for c in row]
    runs = _runs_of_same(cleaned)
    out = []
    for value, _ in runs:
        for _ in range(cells_per_run):
            out.append(value)
    return out


def get_run_lengths_from_header_row(header_row: list) -> list[int]:
    """헤더 행에서 run(연속 같은 값) 길이 목록을 반환."""
    cleaned = [_clean_section_cell(c) for c in header_row]
    runs = _runs_of_same(cleaned)
    return [length for _, length in runs]


def merge_data_row_by_runs(row: list, run_lengths: list[int]) -> list[str]:
    """
    데이터 행을 run 구간별로 잘라, 각 구간의 셀을 한 문자열로 합쳐서 반환.
    run_lengths는 헤더 행에서 get_run_lengths_from_header_row로 구한 값.
    """
    out = []
    start = 0
    for length in run_lengths:
        end = start + length
        chunk = row[start:end] if end <= len(row) else row[start:]
        merged = " ".join(str(c).strip() for c in chunk if str(c).strip())
        out.append(merged)
        start = end
    return out


def _dedupe_words_in_cell(text: str) -> str:
    """띄어쓰기 기준으로 단어를 나누어, 같은 단어 중복 제거(첫 등장만 유지)."""
    if text is None:
        return ""
    s = str(text).strip()
    if not s:
        return ""
    seen = set()
    words = s.split()
    return " ".join(w for w in words if w not in seen and not seen.add(w))

In [35]:
# 재구성 2.2 열분해 (quotation_meta, remarks 추출) + remarks 이동
# 참고: reconstruct()가 테이블 하나일 때 자동 적용. 위 2.1 셀 실행 후 이 셀 실행하면 동일 결과.

from document_parsing_engine.domain.segment_reconstruction import apply_column_merge_and_remarks

refined_seg, refined_blocks = apply_column_merge_and_remarks(refined_seg, refined_blocks)
segment_mapper.pretty_print(refined_seg, refined_blocks)

[#/tables/0@quotation_meta] run_lengths: [2, 3, 3], 행 수: 8 → 8
[#/tables/0@items] run_lengths: [1, 1, 3, 1, 1, 1, 1], 행 수: 19 → 19
=== segment: quotation_meta ===
  block_refs: ['#/tables/0@quotation_meta']
    - #/tables/0@quotation_meta | label=table | content_type=table
       ['견적의뢰정보', '의뢰자정보', '필수요청사항']
       ['상 호 마켓오브메테리얼', '상 호 케이티엘㈜', '1. 세금계산서 발행메일 info@tkdk.co.kr']
       ['수 신 김경호 이사님', '사업자등록번호 621-81-36401 / 대표 : 지춘현', '621-81-36401 / 대표 : 지춘현 2. 화물 납품 대신화물/택배 부산장안점']
       ['견적요청일 2024년 02월 28일', '담당자 지성민', '지성민 3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166']
       ['납품요청일 납품일 별도 회신 요청', '담당자 연락처 051-532-0471 / Sales2@tkdk.co.kr', '051-532-0471 / Sales2@tkdk.co.kr 3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청']
       ['특이사항', '특이사항', '4. 납품요청일에 따른 지체상환금은 납품요청일로부터 7일 이후 적용되며 1일에 3/1000(0.3%)가 적용. 이에 반드시 납품일 기입요청']
       ['자재/부품 요청내역', '자재/부품 요청내역', '자재/부품 요청내역']
       ['최 종 합 계 - ₩', '최 종 합 계 - ₩', '최 종 합 계 - ₩']

=== segment: items ===
  block_re

In [41]:
from document_parsing_engine.app.services.segment_reconstruction_service import SegmentReconstructionService
ss = SegmentReconstructionService()
seg = segment_mapper.recommend(
    doc_type=classification.doc_type.value,  # 또는 doc_type="quotation"
    doc_dict=doc_dict,
    blocks=blocks,
)
rs, rb = ss.reconstruct(seg, blocks)

segment_mapper.pretty_print(rs, rb)

=== segment: quotation_meta ===
  block_refs: ['#/tables/0@quotation_meta']
    - #/tables/0@quotation_meta | label=table | content_type=table
       ['견적의뢰정보', '', '의뢰자정보', '필수요청사항']
       ['상 호 마켓오브메테리얼', '상', '호 케이티엘㈜', '1. 세금계산서 발행메일 info@tkdk.co.kr']
       ['수 신 김경호 이사님', '', '사업자등록번호 621-81-36401 / 대표 : 지춘현', '2. 화물 납품 대신화물/택배 부산장안점']
       ['견적요청일 2024년 02월 28일', '', '담당자 지성민', '3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166']
       ['납품요청일 납품일 별도 회신 요청', '담당자', '연락처 051-532-0471 / Sales2@tkdk.co.kr', '3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청']
       ['특이사항', '', '특이사항', '4. 납품요청일에 따른 지체상환금은 납품요청일로부터 7일 이후 적용되며 1일에 3/1000(0.3%)가 적용. 이에 반드시 납품일 기입요청']
       ['자재/부품 요청내역', '자재/부품 요청내역', '자재/부품 요청내역', '자재/부품 요청내역']
       ['최 종 합 계 - ₩', '최 종 합 계 - ₩', '최 종 합 계 - ₩', '최 종 합 계 - ₩']

=== segment: items ===
  block_refs: ['#/tables/0@items']
    - #/tables/0@items | label=table | content_type=table
       ['NO.', '품 목', '재질 및 규격', '수 량', '단 위', '단 가', '합 계